# Week 2: NLP Preprocessing Pipeline
## TruthLens — Fake News Detector

**Goal**: Prepare text data for machine learning by cleaning, tokenizing, and vectorizing news articles with TF-IDF.

## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import nltk
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
import joblib

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

data_dir = Path('..') / 'data'
model_dir = Path('..') / 'backend' / 'models'
model_dir.mkdir(parents=True, exist_ok=True)

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

## 2. Load and Inspect Data

In [ ]:
real_path = data_dir / 'True.csv'
fake_path = data_dir / 'Fake.csv'

if not real_path.exists() or not fake_path.exists():
    raise FileNotFoundError('True.csv or Fake.csv not found in data directory')

real_df = pd.read_csv(real_path)
fake_df = pd.read_csv(fake_path)

real_df['label'] = 0
fake_df['label'] = 1

# Combine datasets
df = pd.concat([real_df, fake_df], ignore_index=True)

print(f'Loaded real articles: {len(real_df):,}')
print(f'Loaded fake articles: {len(fake_df):,}')
print(f'Total articles: {len(df):,}')
df.head()

## 3. Text Cleaning Functions

In [ ]:
def normalize_text(text: str) -> str:
    if not isinstance(text, str):
        return ''
    text = text.lower()
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def clean_text(text: str) -> str:
    text = normalize_text(text)
    tokens = nltk.word_tokenize(text)
    filtered = [lemmatizer.lemmatize(token) for token in tokens if token not in stop_words and len(token) > 1]
    return ' '.join(filtered)

# Test cleaning on sample text
sample = 'Breaking: Trump visits the White House on Monday, and the crowd was enormous!'
print(clean_text(sample))

## 4. Apply Cleaning to the Dataset

In [ ]:
df['text_clean'] = df['text'].astype(str).apply(clean_text)
df['title_clean'] = df['title'].astype(str).apply(clean_text)
df['combined_text'] = df['title_clean'] + ' ' + df['text_clean']

print(df[['title', 'text_clean', 'combined_text']].head(3).to_string(index=False))
print()
print('Clean text sample length:', df.loc[0, 'text_clean'][:120])

## 5. Feature Engineering with TF-IDF

In [ ]:
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X = tfidf.fit_transform(df['combined_text'])
y = df['label'].values

print('TF-IDF matrix shape:', X.shape)
print('Sample feature names:', tfidf.get_feature_names_out()[:20])

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print('Training shape:', X_train.shape)
print('Test shape:', X_test.shape)

## 6. Save Preprocessing Artifacts

In [ ]:
joblib.dump(tfidf, model_dir / 'tfidf_vectorizer.joblib')
df.to_csv(data_dir / 'cleaned_news.csv', index=False)
print('Saved TF-IDF vectorizer to', model_dir / 'tfidf_vectorizer.joblib')
print('Saved cleaned dataset to', data_dir / 'cleaned_news.csv')

## 7. Observations & Next Steps

- Cleaned text now contains lowercased lemmatized tokens with stopwords removed.
- Combined title and article text gives a strong feature source for TF-IDF.
- TF-IDF produces a feature matrix suitable for model training.
- Next: train classical ML models in Week 3 with logistic regression, random forest, and naive Bayes.